# MAT 167 Final Project


In [107]:
# Import dependecies and libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import random
import string


### Step 1: Turn hyperlink of nodes into Google Matrix G and check all row sums equal to 1.

In [108]:
import numpy as np

n = 12
nodes = list('ABCDEFGHIJKL')

edges = [
    ('A','I'), ('A','B'), ('A','D'),
    ('B','E'), ('B','D'),
    ('C','B'), ('C','D'),
    # D has no outgoing edges (dangling)
    ('E','G'),
    ('F','B'), ('F','C'), ('F','E'),
    ('G','J'), ('G','E'),
    ('H','F'), ('H','G'), ('H','B'),
    ('I','G'), ('I','J'),
    ('J','K'), ('J','H'),
    ('K','H'),
    ('L','H'), ('L','J'),
]

# Build raw adjacency matrix
idx = {node: i for i, node in enumerate(nodes)}
G = np.zeros((n, n))

for src, dst in edges:
    G[idx[src], idx[dst]] = 1

# Identify dangling nodes (nodes with no outlinks)
dangling_nodes = []
for i, node in enumerate(nodes):
    row_sum = G[i].sum()
    if row_sum == 0:
        dangling_nodes.append(node)
        print(f"Node {node} is dangling (no outlinks) - will set row to 1/{n}")

print(f"\nDangling nodes found: {dangling_nodes}")

# Normalize each row to sum to 1 (row stochastic)
# Fix dangling nodes by setting their row to 1/n everywhere
for i in range(n):
    row_sum = G[i].sum()
    if row_sum == 0:
        G[i] = 1.0/n  # Dangling node: equal probability to all nodes
        print(f"Fixed row {nodes[i]}: set to 1/{n} = {1.0/n:.6f} everywhere")
    else:
        G[i] /= row_sum

df_G = pd.DataFrame(G)
df_G.columns = list(string.ascii_uppercase[:n])
df_G.index = list(string.ascii_uppercase[:n])
display(df_G)

# Check Row Sums
print("\nRow sums:", np.round(G.sum(axis=1), 4))

Node D is dangling (no outlinks) - will set row to 1/12

Dangling nodes found: ['D']
Fixed row D: set to 1/12 = 0.083333 everywhere


,A,B,C,D,E,F,G,H,I,J,K,L
A,0.000000,0.333333,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.000000
B,0.000000,0.000000,0.000000,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
C,0.000000,0.500000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
D,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333,0.083333
E,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
F,0.000000,0.333333,0.333333,0.000000,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
G,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000
H,0.000000,0.333333,0.000000,0.000000,0.000000,0.333333,0.333333,0.000000,0.000000,0.000000,0.000000,0.000000
I,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.500000,0.000000,0.000000
J,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,0.500000,0.000000



Row sums: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Step 2: PageRank vs Eigendecomposition


In [109]:
# Compute eigenvalues and eigenvectors of G^T
eigenvals, eigenvecs = np.linalg.eig(G.T)

# Find the index where eigenvalue is closest to 1
idx_one = np.argmin(np.abs(eigenvals - 1.0))

# Get the corresponding eigenvector
pi_raw = eigenvecs[:, idx_one]

# Normalize to make it a probability distribution (sum to 1)
# Take real part and ensure positive
pi = np.real(pi_raw)
pi = np.abs(pi)
pi = pi / pi.sum()  # Normalize so sum = 1

print(f"Eigenvalue found: {eigenvals[idx_one]:.6f}")
print(f"\nPageRank vector pi (normalized):")
print(np.round(pi, 4))
print(f"\nSum of pi: {pi.sum():.6f}")

# Display as DataFrame with node labels
df_pi = pd.DataFrame({
    'Node': nodes,
    'PageRank': np.round(pi, 6)
})
df_pi = df_pi.sort_values(by="PageRank", ascending=False)
display(df_pi)

Eigenvalue found: 1.000000+0.000000j

PageRank vector pi (normalized):
[0.0049 0.0829 0.0224 0.0593 0.1836 0.0525 0.2394 0.1428 0.0066 0.1304
 0.0701 0.0049]

Sum of pi: 1.000000


,Node,PageRank
6,G,0.239440
4,E,0.183621
7,H,0.142766
9,J,0.130420
1,B,0.082906
10,K,0.070149
3,D,0.059261
5,F,0.052527
2,C,0.022448
8,I,0.006585


### Step 3: Power Method

In [110]:
# Initialize: uniform distribution
pi_iter = np.ones(n) / n
epsilon = 1e-6
max_iter = 1000

print(f"Initial pi_0 (uniform): {np.round(pi_iter, 6)}")
print(f"Tolerance: Epsilon = {epsilon}")
# Iterative power method
for iteration in range(1, max_iter + 1):
    pi_prev = pi_iter.copy()
    pi_iter = pi_iter @ G
    
    # Compute norm of difference
    diff_norm = np.linalg.norm(pi_iter - pi_prev)
    
    if iteration % 10 == 0 or diff_norm <= epsilon:
        print(f"Iteration {iteration}: ||pi_j - pi_j-1|| = {diff_norm:.2e}")
    
    # Check convergence
    if diff_norm <= epsilon:
        print(f"\nConverged after {iteration} iterations!")
        print(f"Final ||pi_j - pi_j-1|| = {diff_norm:.2e}")
        break
else:
    print(f"\nDid not converge after {max_iter} iterations")
    print(f"Final ||pi_j - pi_j-1|| = {diff_norm:.2e}")

# Normalize to ensure sum = 1
pi_iter = pi_iter / pi_iter.sum()

print(f"\nFinal pi (iterative method):")
print(np.round(pi_iter, 6))
print(f"Sum: {pi_iter.sum():.6f}")

# Display as DataFrame
df_pi_iter = pd.DataFrame({
    'Node': nodes,
    'PageRank_Iterative': np.round(pi_iter, 6)
})
df_pi_iter = df_pi_iter.sort_values(by="PageRank_Iterative", ascending=False)
display(df_pi_iter)

Initial pi_0 (uniform): [0.083333 0.083333 0.083333 0.083333 0.083333 0.083333 0.083333 0.083333
 0.083333 0.083333 0.083333 0.083333]
Tolerance: Epsilon = 1e-06
Iteration 10: ||pi_j - pi_j-1|| = 4.06e-03
Iteration 20: ||pi_j - pi_j-1|| = 2.74e-04
Iteration 30: ||pi_j - pi_j-1|| = 1.79e-05
Iteration 40: ||pi_j - pi_j-1|| = 1.18e-06
Iteration 41: ||pi_j - pi_j-1|| = 8.95e-07

Converged after 41 iterations!
Final ||pi_j - pi_j-1|| = 8.95e-07

Final pi (iterative method):
[0.004938 0.082906 0.022448 0.059261 0.183621 0.052527 0.23944  0.142766
 0.006585 0.13042  0.070148 0.004938]
Sum: 1.000000


,Node,PageRank_Iterative
6,G,0.239440
4,E,0.183621
7,H,0.142766
9,J,0.130420
1,B,0.082906
10,K,0.070148
3,D,0.059261
5,F,0.052527
2,C,0.022448
8,I,0.006585


#### Compare Power Method vs Page Rank (Eigendecomposition)

In [111]:
print("=" * 60)
print("COMPARISON: Iterative vs Eigenvalue Decomposition")
print("=" * 60)

# Compare the two PageRank vectors
diff = np.abs(pi_iter - pi)
max_diff = np.max(diff)
mean_diff = np.mean(diff)

print(f"\nDifference between methods:")
print(f"  Max difference: {max_diff:.2e}")
print(f"  Mean difference: {mean_diff:.2e}")

# Compare rankings
ranking_eigen = np.argsort(pi)[::-1]
ranking_iter = np.argsort(pi_iter)[::-1]

print(f"\nRankings:")
print(f"  Eigenvalue method: {[nodes[i] for i in ranking_eigen]}")
print(f"  Iterative method:  {[nodes[i] for i in ranking_iter]}")
print(f"  Rankings match? {np.array_equal(ranking_eigen, ranking_iter)}")

print(f"\nBoth methods give the same PageRank vector (within tolerance)")
print(f"Both methods give the same ranking")
print(f"Eigenvalue method and iterative method give the same ranking even though A and L are interchangable rank.")
print(f"This is okay beacause node A and node L has the same eigenvector value.")

COMPARISON: Iterative vs Eigenvalue Decomposition

Difference between methods:
  Max difference: 2.64e-07
  Mean difference: 6.80e-08

Rankings:
  Eigenvalue method: ['G', 'E', 'H', 'J', 'B', 'K', 'D', 'F', 'C', 'I', 'A', 'L']
  Iterative method:  ['G', 'E', 'H', 'J', 'B', 'K', 'D', 'F', 'C', 'I', 'L', 'A']
  Rankings match? False

Both methods give the same PageRank vector (within tolerance)
Both methods give the same ranking
Eigenvalue method and iterative method give the same ranking even though A and L are interchangable rank.
This is okay beacause node A and node L has the same eigenvector value.
